# Does Domain Randomization Actually Help?

**Everyone cites it. Few measure it. Two Isaac Sim datasets that differ in exactly one variable, and an honest look at what the numbers say.**

---

> Part of an open series on running **NVIDIA Isaac Sim on free GPUs**.
> Isaac Sim is free software (Apache 2.0) — only compute ever costs money,
> and this series is about not paying for that either.
>
> If this is useful, an upvote helps other people find it. Questions in the
> comments get answered.

---

## The claim, and the problem with it

**Domain randomization (DR)** is the standard answer to the sim2real gap: if
you randomize the appearance of your simulation aggressively enough — colors,
lighting, textures, camera pose — the real world starts to look like just
another random variation, and a model trained purely in sim transfers.

It is cited constantly. It is also usually asserted rather than measured, and
when people do report it, the comparison is frequently confounded: the DR
dataset has more images, or more object poses, or a different training
schedule.

This notebook runs the comparison properly. Two Isaac Sim datasets:

| | Control | Treatment |
|---|---|---|
| Frames | 2000 | 2000 |
| Object pose randomized | ✅ | ✅ |
| Colors randomized | ✗ | ✅ |
| Lighting randomized | ✗ | ✅ |
| Camera pose randomized | ✗ | ✅ |

**Object pose is randomized in both arms.** That detail matters more than it
looks. If the control arm had static objects, we would be comparing "varied
data vs identical data" and DR would win trivially, for reasons that have
nothing to do with domain randomization. The only variable that differs here
is *appearance*.

Same generator script, same seed policy, same count, same model, same
schedule. Generator source is in the linked dataset if you want to check that
claim rather than take my word for it.

In [ ]:
import json, os, glob
import numpy as np

PLAIN = "/kaggle/input/isaac-sim-synthetic-robot-vision"
DR    = "/kaggle/input/isaac-sim-domain-randomized"

def describe(path, label):
    meta_p = os.path.join(path, "dataset_meta.json")
    if not os.path.exists(meta_p):
        print(f"[!] {label}: not attached. Add the dataset in the right-hand panel.")
        return None
    meta = json.load(open(meta_p))
    n = len(glob.glob(os.path.join(path, "**", "rgb_*.png"), recursive=True))
    print(f"{label:>10}: {n:5d} images | DR={meta['domain_randomization']} "
          f"| classes={meta['classes']}")
    return meta

m_plain = describe(PLAIN, "control")
m_dr    = describe(DR, "treatment")

# Guard: the whole point is that the arms are matched. Check it, do not assume.
if m_plain and m_dr:
    assert m_plain["classes"] == m_dr["classes"], "class sets differ!"
    assert m_plain["n_frames"] == m_dr["n_frames"], "frame counts differ!"
    print("\nArms are matched on class set and frame count.")

## What we are measuring against

Training on synthetic data is easy to evaluate badly. If you test on held-out
*synthetic* images, both arms score near-perfect and you learn nothing — the
model has just memorized the renderer.

The only meaningful test set is **real photographs**. So the evaluation here
is: train on synthetic, test on real. That is the actual sim2real question.

The real test images are a small hand-labeled set of the same four object
classes photographed under ordinary indoor lighting. Small, but the point is
the *difference between arms*, and both arms face the identical test set.

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms, models, datasets

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE=="cuda" else "")

IMSIZE = 128
tf_train = transforms.Compose([
    transforms.Resize((IMSIZE, IMSIZE)),
    transforms.ToTensor(),
])
tf_eval = tf_train   # identical -- no test-time augmentation, no thumb on scale

def make_model(n_classes=4):
    # Deliberately small and trained from scratch. An ImageNet-pretrained
    # backbone would carry real-world priors that muddy the comparison: we
    # want to measure what the SYNTHETIC data teaches, not what ImageNet did.
    m = models.resnet18(weights=None, num_classes=n_classes)
    return m.to(DEVICE)

def train(loader, epochs=8, lr=3e-4, seed=0):
    torch.manual_seed(seed)
    model = make_model()
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    lossf = nn.CrossEntropyLoss()
    model.train()
    for ep in range(epochs):
        tot = correct = 0; running = 0.0
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            out = model(x)
            loss = lossf(out, y)
            loss.backward(); opt.step()
            running += loss.item() * y.size(0)
            correct += (out.argmax(1) == y).sum().item(); tot += y.size(0)
        print(f"  epoch {ep+1}/{epochs}  loss={running/tot:.4f}  train_acc={correct/tot:.3f}")
    return model

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); correct = tot = 0
    per_class = {}
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        pred = model(x).argmax(1)
        correct += (pred == y).sum().item(); tot += y.size(0)
        for c in y.unique():
            mask = y == c
            d = per_class.setdefault(int(c), [0, 0])
            d[0] += (pred[mask] == c).sum().item(); d[1] += int(mask.sum())
    return correct / tot, per_class

## Run both arms

Identical model, identical hyperparameters, identical seed. The only thing
that changes between the two runs is which directory the training images came
from.

In [ ]:
REAL = "/kaggle/input/real-objects-testset"

def run_arm(train_dir, label, seed=0):
    print(f"\n=== {label} ===")
    tr = datasets.ImageFolder(train_dir, transform=tf_train)
    te = datasets.ImageFolder(REAL, transform=tf_eval)
    tr_l = DataLoader(tr, batch_size=64, shuffle=True, num_workers=2)
    te_l = DataLoader(te, batch_size=64, num_workers=2)
    model = train(tr_l, seed=seed)
    acc, per_class = evaluate(model, te_l)
    print(f"  --> REAL-image accuracy: {acc:.3f}")
    return {"label": label, "real_acc": acc, "per_class": per_class,
            "classes": tr.classes}

results = []
for d, lab in [(os.path.join(PLAIN, "images"), "control (no DR)"),
               (os.path.join(DR, "images"), "treatment (DR)")]:
    if os.path.isdir(d):
        results.append(run_arm(d, lab))
    else:
        print(f"[!] missing {d} -- attach the dataset to run this arm")

## Multiple seeds, because one run proves nothing

A single training run comparison is noise. Neural network training varies
substantially across random seeds, and a gap of a few points between two
single runs tells you nothing about whether the intervention worked.

So: three seeds per arm, and we look at the spread as well as the mean. If the
distributions overlap, the honest conclusion is "no measurable effect", and
that is a result worth reporting too.

In [ ]:
SEEDS = [0, 1, 2]
multi = {"control (no DR)": [], "treatment (DR)": []}

for seed in SEEDS:
    for d, lab in [(os.path.join(PLAIN, "images"), "control (no DR)"),
                   (os.path.join(DR, "images"), "treatment (DR)")]:
        if os.path.isdir(d):
            r = run_arm(d, f"{lab} seed={seed}", seed=seed)
            multi[lab].append(r["real_acc"])

for lab, accs in multi.items():
    if accs:
        print(f"{lab:>18}: mean={np.mean(accs):.3f}  std={np.std(accs):.3f}  "
              f"runs={[f'{a:.3f}' for a in accs]}")

In [ ]:
import matplotlib.pyplot as plt

if all(multi.values()):
    fig, ax = plt.subplots(figsize=(8, 5))
    labels = list(multi.keys())
    means = [np.mean(multi[l]) for l in labels]
    stds  = [np.std(multi[l]) for l in labels]
    bars = ax.bar(labels, means, yerr=stds, capsize=10,
                  color=["#8899aa", "#2a9d8f"], width=0.55)
    for l, x in zip(labels, range(len(labels))):
        ax.scatter([x] * len(multi[l]), multi[l], color="k", zorder=3, s=35,
                   label="individual seeds" if x == 0 else None)
    ax.set_ylabel("accuracy on REAL photographs")
    ax.set_title("Sim-to-real transfer: does domain randomization help?")
    ax.set_ylim(0, 1); ax.grid(axis="y", alpha=0.3); ax.legend()
    plt.tight_layout(); plt.show()

    delta = np.mean(multi["treatment (DR)"]) - np.mean(multi["control (no DR)"])
    pooled = np.sqrt(np.mean([np.var(v) for v in multi.values()]))
    print(f"\nabsolute difference : {delta:+.3f}")
    print(f"pooled std          : {pooled:.3f}")
    print(f"effect size (Cohen d): {delta/pooled:.2f}" if pooled > 0 else "")

## Reading the result honestly

Whatever the numbers came out as, here is how to interpret them without
overclaiming — which is most of the value of running a controlled experiment
in the first place.

**If DR wins by a large margin relative to the seed spread**, that is the
expected result and it supports the standard advice. Note that the mechanism
is specific: DR does not make the model *better*, it makes the model *stop
relying on color and lighting cues* that do not survive the transfer to real
images. You are removing a shortcut, not adding capability.

**If the arms overlap**, that is a real and publishable finding too. Common
reasons this happens:
- The test set is too small for the difference to be resolvable
- The object classes are distinguishable by *shape* alone, so appearance
  randomization was never load-bearing
- The randomization ranges were too narrow to matter, or so wide that they
  destroyed the signal

That last failure mode deserves emphasis: **DR is not monotonic.** Randomize
too aggressively and real images fall *outside* the training distribution
rather than inside it, and transfer gets worse. The commonly-repeated advice
"just randomize everything" is wrong at the limit.

### What this notebook is not

A single object set, a small real test set, one architecture. This is a
controlled experiment, not a general law about domain randomization. The
method is the transferable part — matched arms, multiple seeds, real test
images, effect size reported alongside the mean.

If you rerun this with different classes or wider randomization ranges,
**post what you get in the comments.** Negative results especially — those are
the ones nobody publishes and everybody needs.

*Datasets, generator source, and the Isaac Sim Replicator config are all in
the linked datasets. Notebook 1 in this series covers getting a free GPU to
regenerate them yourself.*

---

## Reproducing this

Every notebook in this series runs on free infrastructure. Nothing here needs
a paid GPU.

| Method | Free allowance | Best for |
|---|---|---|
| Kaggle | 30 GPU hr/week, 2x T4 | Running this notebook as-is |
| Lightning AI | 80 GPU hr/month, persistent disk | Heavy Isaac Sim generation |
| Google Colab | best-effort T4 | Quick smoke tests |
| NVIDIA DLI | free hosted labs | Learning the Isaac Sim GUI |

**The one gotcha worth remembering:** Isaac Sim needs **RT cores**. A T4, L4,
L40S or any RTX card is fine. An **A100 or H100 is not** — those have no RT
cores, so the RTX renderer is unsupported or unusably slow. It is the most
counterintuitive constraint in cloud robotics simulation, and it bites people
who assume the more expensive GPU must be the better one.

*Series index and full source: see the linked dataset description.*